# S31 strong judge on Kaggle (2x T4)

Scores the S31 cosine pool (top-2.0 x K_q, about 430k pairs) with a Qwen3 yes/no judge.

**Settings**: Accelerator = GPU T4 x2, Internet = On. No HF token or other secret is needed or used
(all models are public). Run with *Save Version -> Save & Run All (Commit)* so it can run up to 12 h.

**Inputs** (Add Input):
1. Dataset with the pool export: `shards/pool-*.parquet`, `manifest.json`, `s29_qwen06_scores.parquet`, `judge_score.py`.
2. Optional: output of an earlier run of this notebook (same model) to resume; list its path in `RESUME`.

Steps: environment -> 0.6B regression gate vs S29 (Spearman >= 0.99, fp16) -> pilot 1,000 pairs (speed, ETA)
-> run `SUBSET`, one worker per GPU, stops before the time limit -> collect `SUBSET` when every part exists.
Session 1 scores `SUBSET = "pool13"` (the 1.3x pool, enough for J4-rrf / J4-rrf3). A later session sets
`SUBSET = "tail"` (the 1.3x-2.0x pairs), `RUN_GATE = RUN_PILOT = False`, and lists the earlier output in `RESUME`;
when the tail is complete the notebook also collects `all` (for J4-rrf-pool20).
Finished parts are in `/kaggle/working/s31/<model>/parts/`; a partial run is resumed by adding this
notebook's output as an input and listing it in `RESUME`.

In [ ]:
# ---- configuration ----
MODEL = "qwen3-reranker-4b"      # qwen3-reranker-4b | qwen3-reranker-8b | qwen3-14b-awq (vllm)
REVISION = None                  # pin to the commit printed by the download cell for every later session
DTYPE = "float16"                # T4 has no native bf16
BACKEND = "hf"                   # "vllm" only after its pilot matches the hf pilot
SUBSET = "pool13"                # session 1: "pool13" (the 1.3x pool, ~282k pairs); later sessions: "tail" (1.3x-2.0x)
RESUME = []                      # e.g. ["/kaggle/input/s31-4b-run1/s31"]
OUT = "/kaggle/working/s31"
MAX_BATCH_TOKENS = 16384
MAX_SECONDS = 11 * 3600 - 2400   # stop starting parts ~40 min before the 12 h limit (load + collect margin)
RUN_GATE, RUN_PILOT, RUN_FULL = True, True, True
WORKERS = 2                      # one process per T4

In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
import torch, transformers, pandas, pyarrow
print("torch", torch.__version__, "transformers", transformers.__version__, "pandas", pandas.__version__, "pyarrow", pyarrow.__version__)
from packaging.version import Version
if Version(transformers.__version__) < Version("4.51.0"):  # Qwen3 support
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.51.0"], check=True)
    print("upgraded transformers; restart the kernel if the import above was already done")
manifests = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert len(manifests) == 1, manifests
POOL = str(Path(manifests[0]).parent)
SCRIPT = glob.glob(f"{POOL}/**/judge_score.py", recursive=True)[0]
REF = glob.glob(f"{POOL}/**/s29_qwen06_scores.parquet", recursive=True)[0]
manifest = json.load(open(manifests[0]))
print("pool", POOL, "pairs", manifest["pairs"], "shards", len(manifest["shards"]))
os.makedirs(OUT, exist_ok=True)

In [ ]:
# Download once (public model, no token) so the two workers do not race on the cache.
from huggingface_hub import snapshot_download
specs = {"qwen3-reranker-0.6b": ("Qwen/Qwen3-Reranker-0.6B", "e61197ed45024b0ed8a2d74b80b4d909f1255473"),
         "qwen3-reranker-4b": ("Qwen/Qwen3-Reranker-4B", REVISION),
         "qwen3-reranker-8b": ("Qwen/Qwen3-Reranker-8B", REVISION),
         "qwen3-14b": ("Qwen/Qwen3-14B", REVISION),
         "qwen3-14b-awq": ("Qwen/Qwen3-14B-AWQ", REVISION)}
t0 = time.time()
repo_id, revision = specs[MODEL]
path = snapshot_download(repo_id=repo_id, revision=revision)  # keyword-only in recent huggingface_hub
RESOLVED = Path(path).name
print(MODEL, "snapshot", path, "commit", RESOLVED, f"{time.time() - t0:.0f}s")
if RUN_GATE:
    gate_repo, gate_revision = specs["qwen3-reranker-0.6b"]
    snapshot_download(repo_id=gate_repo, revision=gate_revision)

In [ ]:
def run(args, gpu="0"):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": gpu, "TOKENIZERS_PARALLELISM": "false"}
    cmd = [sys.executable, SCRIPT, *args]
    print(" ".join(cmd))
    subprocess.run(cmd, env=env, check=True)

common = ["--pool", POOL, "--out", OUT, "--dtype", DTYPE, "--max-batch-tokens", str(MAX_BATCH_TOKENS)]
if RUN_GATE:  # 0.6B regression vs S29 scores (same prompt, fp16 here vs bf16 in S29)
    run(["pilot", *common, "--model", "qwen3-reranker-0.6b", "--pairs", "1000"])
    ours = pandas.read_parquet(f"{OUT}/qwen3-reranker-0.6b/pilot-1000-hf-{DTYPE}.parquet")
    ref = pandas.read_parquet(REF)
    j = ours.merge(ref[["pair_id", "qwen_score"]], on="pair_id")
    rho = j.score.corr(j.qwen_score, method="spearman")
    gate = {"pairs": len(j), "spearman": rho, "abs_diff_max": float((j.score - j.qwen_score).abs().max())}
    json.dump(gate, open(f"{OUT}/gate-0.6b.json", "w"), indent=2)
    print(gate)
    assert len(j) == 1000 and rho >= 0.99, "regression gate failed: do not run the full pool"

In [ ]:
if RUN_PILOT:
    rev = ["--revision", RESOLVED] if MODEL != "qwen3-reranker-0.6b" else []
    run(["pilot", *common, "--model", MODEL, "--backend", BACKEND, "--pairs", "1000", *rev])
    pilot = json.load(open(f"{OUT}/{MODEL}/pilot-1000-{BACKEND}-{DTYPE}.json"))
    rate = pilot["pairs_per_second"]
    print(json.dumps({k: pilot[k] for k in ("pairs_per_second", "seconds", "prompt_tokens_mean", "peak_cuda_allocated_mib", "gpu") if k in pilot}, indent=2))
    sizes = {"all": manifest["pairs"], "pool13": manifest["pool13_pairs"], "tail": manifest["pairs"] - manifest["pool13_pairs"]}
    print(f"ETA subset {SUBSET} ({sizes[SUBSET]} pairs) with {WORKERS} GPUs: {sizes[SUBSET] / (rate * WORKERS) / 3600:.1f} h")

In [ ]:
if RUN_FULL:
    rev = ["--revision", RESOLVED] if MODEL != "qwen3-reranker-0.6b" else []
    resume = [x for r in RESUME for x in ("--resume-from", r)]
    procs = []
    for w in range(WORKERS):
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(w), "TOKENIZERS_PARALLELISM": "false"}
        cmd = [sys.executable, SCRIPT, "score", *common, "--model", MODEL, "--backend", BACKEND, "--subset", SUBSET, *rev, *resume,
               "--worker", str(w), "--workers", str(WORKERS), "--max-seconds", str(MAX_SECONDS)]
        procs.append(subprocess.Popen(cmd, env=env, stdout=open(f"{OUT}/worker-{w}-{SUBSET}.stdout", "w"), stderr=subprocess.STDOUT))
    while any(p.poll() is None for p in procs):
        time.sleep(600)
        for w in range(WORKERS):
            lines = open(f"{OUT}/worker-{w}-{SUBSET}.stdout").read().splitlines()
            print(w, lines[-1][:300] if lines else "")
    print("exit codes", [p.returncode for p in procs])

In [ ]:
def status(subset):
    resume = [x for r in RESUME for x in ("--resume-from", r)]
    res = subprocess.run([sys.executable, SCRIPT, "status", "--pool", POOL, "--out", OUT, "--model", MODEL, "--subset", subset, *resume],
                         capture_output=True, text=True, check=True)
    return json.loads(res.stdout.strip().splitlines()[-1])

st = status(SUBSET)
print(st)
if st["complete"]:
    run(["collect", "--pool", POOL, "--out", OUT, "--model", MODEL, "--subset", SUBSET])
    if SUBSET != "all" and status("pool13")["complete"] and status("tail")["complete"]:
        run(["collect", "--pool", POOL, "--out", OUT, "--model", MODEL, "--subset", "all"])
else:
    print("partial run: add this notebook's output as input and list it in RESUME for the next session")